In [2]:
# Import the Dash components used to build the application
from dash import Dash, dcc, html, Input, Output

# Import the MongoDB data-access class
from data_access.CRUD_Python_Module import AnimalShelter

# Imports the dashboard layouts and callbacks
from app.Dashboard_Layout import create_dashboard_layout
from app.Dashboard_Callbacks import register_callbacks
from app.Admin_Layout import create_admin_layout
from app.Admin_Callbacks import register_admin_callbacks

# Imports the functions used for animal records and default rescue profiles
from services.Animal_Service import get_animals_by_type
from services.Rescue_Profiles import get_default_profiles

# Application configuration
APP_PORT = 8051

# Test credentials that are used to connect to MongoDB
username = "aacuser"
password = "testpassword"

# Application Setup:
# Create the connection to the animal shelter database
shelter = AnimalShelter(
    username,
    password
)

# Load the initial shelter records for the dashboard
initial_dataframe = get_animals_by_type(
    shelter,
    "all"
)

# Create the Dash application:
# Suppress callback exceptions because each route uses different components
app = Dash(
    __name__,
    suppress_callback_exceptions=True
)

# Application Shell:
app.layout = html.Div([

    # Tracks the current page in the application
    dcc.Location(
        id="url",
        refresh=False
    ),

    # Store rescue-profile changes for the current browser session
    # Note: Persistent MongoDB storage will be added in the database enhancement
    dcc.Store(
        id="rescue-profile-store",
        data=get_default_profiles(),
        storage_type="session"
    ),

    # Display the layout associated with the current route
    html.Div(
        id="page-content"
    )
])

# Application Routing:
@app.callback(
    Output(
        "page-content",
        "children"
    ),
    Input(
        "url",
        "pathname"
    )
)

def display_page(pathname):

    # Display the rescue profile configuration page if the user navigates to the /admin route
    if pathname == "/admin":
        return create_admin_layout()

    # Display the main Animal Shelter Dashboard
    if pathname == "/":
        return create_dashboard_layout(
            initial_dataframe
        )

    # Display a message when the user navigates to a route that does not exist
    return html.Div([
        html.H1(
            "Page Not Found"
        ),

        dcc.Link(
            "Return to Dashboard",
            href="/"
        )
    ])

# Register callbacks used by the main dashboard
register_callbacks(
    app,
    shelter
)

# Register callbacks used by the rescue-profile configuration page
register_admin_callbacks(
    app
)

# Run the application:
# Start the dashboard using the configured local port
app.run(
    jupyter_mode="external",
    port=APP_PORT
)

Dash app running on http://127.0.0.1:8051/
